#STURM-Flood

In [1]:
#@title Imports

import random
import os
import sys
import zipfile
import ee
import time

from dataclasses import dataclass
from pathlib import Path
from google.colab import auth, drive

In [2]:
#@title Google auth + Mount Drive + Optional clone

root_path = "/content/drive/MyDrive/MSc/STURM-fusion"  #@param {type:"string", multiline:true}
mount_point = "/content/drive"  #@param {type:"string"}
clone_repo = False  #@param {type:"boolean"}
reset_export = False  #@param {type:"boolean"}
gee_export = False  #@param {type:"boolean"}
push_to_HF = False  #@param {type:"boolean"}
cancle_gee_tasks = False  #@param {type:"boolean"}
gee_project = "243624085884"  #@param {type:"string"}

# Always authenticate + mount
drive.mount(mount_point)


ee.Authenticate(force=True, auth_mode="notebook")
ee.Initialize(project=gee_project)

repo_url = "https://github.com/TAX2310/STURM-fusion.git"

if clone_repo:
    project_root = os.path.join(root_path, "STURM-fusion")
    if not os.path.exists(project_root):
        !git clone {repo_url} "{project_root}"
else:
    project_root = root_path

sys.path.append(project_root)

from config.config import CFG
cfg = CFG()
cfg.ROOT = Path(project_root)
cfg.DRIVE_ROOT = Path(mount_point) / "MyDrive"
cfg.GEE_PROJECT = gee_project

print("✅ ROOT:", cfg.ROOT)
print("✅ DRIVE_ROOT:", cfg.DRIVE_ROOT)
print("✅ GEE project:", cfg.GEE_PROJECT)

Mounted at /content/drive
To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=rhjGGKS4yfiArR4ZdI16lpqc-DVMvSCtUwjumrSRcpE&tc=l3PzOogjZq0I1Pw81VgXVuMG20BaOPQ2NuU3MIqNLwI&cc=S5JhUqDT0q2jSyRyZjZxW7AjaTCqnHni_UrCFplaqYY

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1Aci98E_tORM4uFVAu9xrW4gbeIoieva0Qgp92gbx0fEIUQOQe1McWcIeADI

Successfully saved authorization token.
✅ ROOT: /content/drive/MyDrive/MSc/STURM-fusion
✅ DRIVE_ROOT: /content/drive/MyDrive
✅ GEE project: 243624085884


In [3]:
from src.gee.tasks import cancel_all_tasks
from src.utils.io import clear_export_folder, create_dataset_structure, zip_dataset
from src.data.sturm_flood import download_and_extract
from src.gee.tasks import has_active_tasks
from src.pipeline.build_dataset import process_csv, save_dataframe_to_csv, export_all_s1_images, assemble_dataset, preprocessing_s1_pipeline
from src.pipeline.data_validation import validate_files, validate_nan_files, retry_export_of_nan_files, validate_dataset, check_image_shapes, get_band_min_max, get_band_percentiles, get_max_time_difference_with_row
from src.hugging_face.push_dataset import push_zip_to_hf

In [4]:
if cancle_gee_tasks:
  cancel_all_tasks()

In [5]:
if reset_export:
    clear_export_folder(cfg)

create_dataset_structure(cfg)

✅ Dataset structure created:
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-flood
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24/Dataset
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24/Dataset/S1
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24/Dataset/S2
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24/Dataset/floodmaps
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24/Dataset/metadata


In [6]:
#@title Download Dataset

download_and_extract(cfg)

✅ Zip already exists or dataset present, skipping download.
✅ Dataset already extracted, skipping unzip.


In [7]:
if gee_export and not has_active_tasks():
    print("No active GEE tasks.starting 🚀"  )
    images, df_fusion = process_csv(cfg.OLD_S2_METADATA_CSV, cfg)
    print('')
    print(len(images), "images processed.")

    save_dataframe_to_csv(df_fusion, cfg.NEW_METADATA_CSV)
    print('New Metadata saved')

    export_all_s1_images(images, cfg)
    print('S1 images exported')
else:
    print("Active GEE tasks detected. Please wait for them to finish before startingt. ⏳")

Active GEE tasks detected. Please wait for them to finish before startingt. ⏳


In [ ]:
while not validate_files(cfg):
    assemble_dataset(cfg)
    print('Dataset assembled')

    preprocessing_s1_pipeline(cfg)
    print('S1 preprocessing done')

    time.sleep(120)


📊 Validation Results:
Total rows: 1970
Missing S2: 0
Missing S1: 259
❌ Dataset incomplete
Copying Matched S2 images... 🚀
Copying 1970/1970: EMSR629_AOI02_01_13_2_2.tif
✅ Copied: 0
⏭️ Skipped existing: 1970
⚠️ Missing source: 0
Copying Matched S1 images... 🚀
Copying 1867/1970: EMSR570_AOI01_7_1_1_2.tif

In [ ]:
if not validate_nan_files(cfg):
    df = retry_export_of_nan_files(cfg)

In [ ]:
if validate_dataset(cfg):

    print('Starting inspection Pipeline')

    result = check_image_shapes(cfg.NEW_S1_PATH, cfg.NEW_S2_PATH)
    print(result)

    get_band_min_max(cfg.NEW_S1_PATH)
    get_band_percentiles(cfg.NEW_S1_PATH)

    max_s2_diff = get_max_time_difference_with_row(cfg.NEW_METADATA_CSV, sentinel_timestamp="sentinel2_timestamp")

    print(f"Max S2 time difference: {max_s2_diff['time_diff_hours']:.2f} hours")

    max_s1_diff = get_max_time_difference_with_row(cfg.NEW_METADATA_CSV, sentinel_timestamp="sentinel1_timestamp")

    print(f"Max S1 time difference: {max_s1_diff['time_diff_hours']:.2f} hours")

In [ ]:
if push_to_HF and validate_dataset(cfg):
    zip_dataset(cfg)
    print('Dataset zipped')
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")

    from huggingface_hub import login

    login(token=HF_TOKEN)

    url = push_zip_to_hf(zip_path=cfg.NEW_ZIP_PATH,repo_id=cfg.HF_REPO_ID,path_in_repo="Dataset.zip",private=False,)
    print(url)

In [ ]:
#cancel_all_tasks()